In [1]:
import pandas as pd
import os
import json
import copy
import re
import torch
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from pathlib import Path
from datasets import load_dataset
from openai import OpenAI
from dotenv import load_dotenv

In [42]:
load_dotenv()

DATA_PATH = Path("./data")
RAW_1P_DATA_PATH = DATA_PATH / "input" / "aei_raw_1p_api_2026-02-05_to_2026-02-12.csv"
RAW_CLAUDE_DATA_PATH = DATA_PATH / "input" / "aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv"
PROMPTS_DIR = Path("./prompts")
WORK_RELATED_PROMPT_PATH = PROMPTS_DIR / "work_related.json"

OUTPUT_DIR = DATA_PATH / "output"
WORK_RELATED_OUTPUT_PATH = OUTPUT_DIR / "work_related.csv"
HIERARCHY_OUTPUT_PATH = OUTPUT_DIR / "hierarchy.csv"
TASK_MAPPING_OUTPUT_PATH = OUTPUT_DIR / "task_mapping.csv"

WILDCHAT_DATASET = "allenai/WildChat-4.8M"
CONVERSATION_FIELD = "conversation"
LANGUAGE_FIELD = "language"
TARGET_LANGUAGE = "English"

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

RANDOM_SEED = 42
BUFFER_SIZE = 10_000
SAMPLE_SIZE = 10
MODEL_ID = "nvidia/nemotron-3-super-120b-a12b:free"
EMBEDDING_MODEL_ID = "sentence-transformers/all-mpnet-base-v2"
EMBEDDING_BATCH_SIZE = 256

PARENT_COLUMN = "parent_id"
COUNT_VARIABLE = "onet_task_count"

In [3]:
device = ""

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [4]:
wildchat_ds = load_dataset(
    WILDCHAT_DATASET,
    split="train",
    streaming=True
)
english_conversations_rows = wildchat_ds.filter(lambda x: x[LANGUAGE_FIELD] == TARGET_LANGUAGE)
sample_conversations_rows = pd.DataFrame(english_conversations_rows.shuffle(
    seed=RANDOM_SEED,
    buffer_size=BUFFER_SIZE
).take(SAMPLE_SIZE))
sample_conversations = sample_conversations_rows[[CONVERSATION_FIELD]]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

In [5]:
sample_df = pd.DataFrame(sample_conversations.head())
sample_df.head()

,conversation
0,[{'content': 'i own a rusty 1975 plymouth fury...
1,[{'content': 'Make the first part of the Delta...
2,[{'content': 'How to be a good police officer ...
3,[{'content': 'freedom planet all characters re...
4,[{'content': 'what to positively answer to fol...


In [6]:
client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

In [7]:
def get_gpt_response(messages: list[dict[str, str]]) -> str:
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
    )
    dirty_result = response.choices[0].message.content.strip()
    cleaned_result = re.search(r'<answer>(.*?)</answer>', dirty_result, re.DOTALL)
    if cleaned_result:
        return cleaned_result.group(1)
    return ""


In [8]:
def get_messages(path: Path) -> list[dict[str, str]]:
    with open(path, "r") as f:
        return json.load(f)

In [9]:
def format_conversation(conversation: list[dict[str, str]]) -> str:
    formatted = ""
    for message in conversation:
        role = message["role"]
        content = message["content"]
        formatted += f"{role}: {content}\n"
    return formatted

In [10]:
def build_messages(template: list[dict[str, str]], **kwargs):
    msgs = copy.deepcopy(template)
    for msg in msgs:
        for key, value in kwargs.items():
            if "{" + key + "}" in msg["content"]:
                msg["content"] = msg["content"].replace("{" + key + "}", value)
    return msgs

In [36]:
def filter_work_conversations(conversations: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame()
    template = get_messages(WORK_RELATED_PROMPT_PATH)
    for index, row in conversations.iterrows():
        raw_conversation = row["conversation"]
        conversation = format_conversation(conversation=raw_conversation)
        messages = build_messages(template=template, conversation=conversation)
        response = get_gpt_response(messages=messages)
        record = [{"conversation": conversation, "is_work_related": response}]
        result = pd.concat([result, pd.DataFrame(record)], ignore_index=True)
    return result

In [38]:
if not WORK_RELATED_OUTPUT_PATH.exists():
    work_related_conversations_df = filter_work_conversations(conversations=sample_conversations)
    work_related_conversations_df.to_csv(WORK_RELATED_OUTPUT_PATH)
else:
    work_related_conversations_df = pd.read_csv(WORK_RELATED_OUTPUT_PATH)
    work_related_conversations_df = work_related_conversations_df[work_related_conversations_df["is_work_related"] == "Yes"]
work_related_conversations_df.head()

,Unnamed: 0,conversation,is_work_related
1,1,user: Make the first part of the Delta Systems...,Yes
2,2,user: How to be a good police officer \nassist...,Yes
4,4,user: what to positively answer to following j...,Yes
5,5,user: 翻成英文：猫为什么会在天上飞\nassistant: Why do cats f...,Yes
6,6,user: Rewrite and format the following comment...,Yes


In [26]:
raw_1p_api_df = pd.read_csv(RAW_1P_DATA_PATH)
raw_claude_api_df = pd.read_csv(RAW_CLAUDE_DATA_PATH)

In [27]:
raw_claude_api_df.head()

,geo_id,geography,date_start,date_end,platform_and_product,facet,level,variable,cluster_name,value
0,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_count,directive,15.000000
1,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_pct,directive,20.270270
2,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_count,learning,20.000000
3,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_pct,learning,27.027027
4,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_count,not_classified,16.000000


In [28]:
raw_claude_api_df["cluster_name"] = raw_claude_api_df["cluster_name"].str.split("::").str[0]
columns_to_keep = ["level", "cluster_name"]
clusters_df = (
    raw_claude_api_df[columns_to_keep]
   .drop_duplicates()
   .reset_index(drop=True)
   .iloc[50:]
)

clusters_df.reset_index(inplace=True, drop=True)
clusters_df.head()

,level,cluster_name
0,0,"present investment information, such as produc..."
1,0,"provide advice to clients on a contract basis,..."
2,0,"provide clients with communication assistance,..."
3,0,provide feedback and recommendations to develo...
4,0,provide information and advice to the public r...


In [29]:
model = SentenceTransformer(EMBEDDING_MODEL_ID).to(device=device)

# TODO: Store the embeddings, so that we don't have to recompute them every time.
embeddings = model.encode(
    clusters_df["cluster_name"].to_list(),
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)
clusters_df["embedding"] = list(embeddings)
clusters_df["id"] = clusters_df.index

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [30]:
level_0 = clusters_df[clusters_df["level"] == 0].copy()
level_1 = clusters_df[clusters_df["level"] == 1].copy()
level_2 = clusters_df[clusters_df["level"] == 2].copy()

level_2[PARENT_COLUMN] = None

In [31]:
level_2.head()

,level,cluster_name,embedding,id,parent_id
160,2,"Assist with SQL databases, data analysis, and ...","[-0.03694914, 0.019519594, -0.059328992, -0.01...",160,None
161,2,"Assist with academic research, scientific writ...","[0.051645294, -0.03381015, -0.007419883, -0.04...",161,None
162,2,"Assist with business development, analysis, re...","[0.031849258, 0.0071852207, -0.042096466, -0.0...",162,None
163,2,"Assist with civic information, government serv...","[-0.0074762586, 0.091707915, -0.01566061, 0.00...",163,None
164,2,"Assist with creative content, marketing, and m...","[0.046204854, 0.011296493, -0.030031018, -0.07...",164,None


In [32]:
def assign_parents(parents: pd.DataFrame, children: pd.DataFrame):
    children_matrix = np.stack(children["embedding"])
    parent_matrix = np.stack(parents["embedding"])
    similarity_matrix = cosine_similarity(children_matrix, parent_matrix)
    most_similar_parent_indices = similarity_matrix.argmax(axis=1)
    children[PARENT_COLUMN] = parents["id"].iloc[most_similar_parent_indices].values
    return children

In [33]:
level_0 = assign_parents(parents=level_1, children=level_0)
level_1 = assign_parents(parents=level_2, children=level_1)
hierarchy_df = pd.concat([level_0, level_1, level_2], ignore_index=True)
hierarchy_df.drop(columns=["embedding"], inplace=True)
hierarchy_df.to_csv(HIERARCHY_OUTPUT_PATH, index=False)
hierarchy_df.head()

,level,cluster_name,id,parent_id
0,0,"present investment information, such as produc...",0,147
1,0,"provide advice to clients on a contract basis,...",1,152
2,0,"provide clients with communication assistance,...",2,124
3,0,provide feedback and recommendations to develo...,3,94
4,0,provide information and advice to the public r...,4,147


In [52]:
def get_task_id(task_name: str, hierarchy: pd.DataFrame) -> int:
    record = hierarchy[hierarchy["cluster_name"] == task_name]
    if not record.empty:
        return record["id"].values[0]
    return None

In [53]:
def format_options(tasks: pd.DataFrame) -> str:
    options_str = ""
    for _, row in tasks.iterrows():
        option_name = row["cluster_name"]
        options_str += f"{option_name}\n"
    return options_str

In [67]:
def map_conversation_to_task(conversations: pd.DataFrame, tasks: pd.DataFrame) -> pd.DataFrame:
    template = get_messages(PROMPTS_DIR / "task_mapping.json")
    result = pd.DataFrame(columns=["conversation", "level_2_task", "level_1_task", "level_0_task"])

    for index, row in conversations.iterrows():
        raw_conversation = row["conversation"]
        conversation = format_conversation(conversation=raw_conversation)
        record = {"conversation": conversation, "level_2_task": None, "level_1_task": None, "level_0_task": None}

        high_level_tasks = tasks[tasks["level"] == 2]
        options_str = format_options(tasks=high_level_tasks)
        messages = build_messages(template=template, conversation=conversation, options_str=options_str)
        level_2_response = get_gpt_response(messages=messages).strip()
        record["level_2_task"] = level_2_response

        parent_id = get_task_id(task_name=level_2_response, hierarchy=tasks)
        medium_level_tasks = tasks[(tasks["level"] == 1) & (tasks["parent_id"] == parent_id)]
        options_str = format_options(tasks=medium_level_tasks)
        messages = build_messages(template=template, conversation=conversation, options_str=options_str)
        level_1_response = get_gpt_response(messages=messages).strip()
        record["level_1_task"] = level_1_response

        parent_id = get_task_id(task_name=level_1_response, hierarchy=tasks)
        low_level_tasks = tasks[(tasks["level"] == 0) & (tasks["parent_id"] == parent_id)]
        options_str = format_options(tasks=low_level_tasks)
        messages = build_messages(template=template, conversation=conversation, options_str=options_str)
        level_0_response = get_gpt_response(messages=messages).strip()
        record["level_0_task"] = level_0_response

        result = pd.concat([result, pd.DataFrame([record])], ignore_index=True)

    return result

In [68]:
if not TASK_MAPPING_OUTPUT_PATH.exists():
    mapped_conversations_df = map_conversation_to_task(conversations=sample_conversations, tasks=hierarchy_df)
    mapped_conversations_df.to_csv(TASK_MAPPING_OUTPUT_PATH, index=False)
else:
    mapped_conversations_df = pd.read_csv(TASK_MAPPING_OUTPUT_PATH)
mapped_conversations_df.head()

,conversation,level_2_task,level_1_task,level_0_task
0,user: i own a rusty 1975 plymouth fury beater....,not_classified,Provide emotional support,Providing emotional support
1,user: Make the first part of the Delta Systems...,"Assist with creative content, marketing, and m...","Assist with creative fiction writing, editing,...",Get creative writing assistance for fiction de...
2,user: How to be a good police officer \nassist...,"Assist with job applications, career transitio...",Provide comprehensive career development and j...,advise students on academic and vocational cur...
3,user: freedom planet all characters react to g...,"Assist with creative content, marketing, and m...","Assist with creative fiction writing, editing,...",Write and develop fanfiction stories based on ...
4,user: what to positively answer to following j...,"Assist with job applications, career transitio...","Create and optimize resumes, cover letters, an...",Draft and refine cover letters for job applica...
